In [ ]:
import pyspark
import os
import sys
from pyspark.sql import SparkSession

os.environ['HADOOP_USER_NAME']="anonymous"

jars_dir = "/tmp/gravitino/spark/packages/"
all_jars = [os.path.join(jars_dir, f) for f in os.listdir(jars_dir) if f.endswith(".jar")]
jars_str = ",".join(all_jars)

spark = None

try:
    spark = SparkSession.builder \
        .appName("PySpark SQL Example") \
        .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
        .config("spark.jars", jars_str) \
        .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
        .config("spark.sql.gravitino.metalake", "metalake_demo") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions") \
        .config("spark.sql.catalog.catalog_rest", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.catalog_rest.type", "rest") \
        .config("spark.sql.catalog.catalog_rest.uri", "http://gravitino:9001/iceberg/") \
        .config("spark.sql.catalog.catalog_rest.warehouse", "hdfs://hive:9000/user/iceberg/warehouse/") \
        .config("spark.sql.catalog.catalog_hive", "org.apache.gravitino.spark.connector.hive.GravitinoHiveCatalogSpark34") \
        .config("spark.sql.catalog.catalog_hive.spark.sql.hive.metastore.jars.path", "file:///opt/spark/jars/*") \
        .config("spark.sql.catalog.paimon", "org.apache.paimon.spark.SparkCatalog") \
        .config("spark.sql.catalog.paimon.warehouse", "hdfs://hive:9000/user/hive/warehouse/") \
        .config("spark.locality.wait.node", "0") \
        .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
        .enableHiveSupport() \
        .getOrCreate()
    
    print("SparkSession successfully created!")

except Exception as e:
    print("Error when create SparkSession:", e, file=sys.stderr)
    if spark is not None:
        spark.stop()
    sys.exit(1)

In [2]:
spark.sql("use catalog_paimon")
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
|    sales|
+---------+



In [10]:
spark.sql("CREATE DATABASE IF NOT EXISTS mydatabase;")
spark.sql("USE mydatabase;")

DataFrame[]

In [15]:
spark.sql("""
CREATE TABLE IF NOT EXISTS employee (
  id BIGINT,
  name STRING,
  department STRING,
  hire_date TIMESTAMP
)
USING paimon
PARTITIONED BY (name)
TBLPROPERTIES (
  'metastore'='hive'
);
""")
spark.sql("SHOW TABLES;").show()

+----------+---------+-----------+
| namespace|tableName|isTemporary|
+----------+---------+-----------+
|mydatabase| employee|      false|
+----------+---------+-----------+



In [16]:
spark.sql("""
INSERT INTO employee
VALUES
(1, 'Alice', 'Engineering', TIMESTAMP '2021-01-01 09:00:00'),
(2, 'Bob', 'Marketing', TIMESTAMP '2021-02-01 10:30:00'),
(3, 'Charlie', 'Sales', TIMESTAMP '2021-03-01 08:45:00');
""")

DataFrame[]

In [17]:
spark.sql("DROP TABLE employee;")

DataFrame[]

Restart spark environment

In [1]:
from pyspark.sql import SparkSession
import os

os.environ['HADOOP_USER_NAME']="anonymous"

spark = SparkSession.builder \
    .appName("PaimonDemoNotebook") \
    .config("spark.jars", "/tmp/gravitino/spark/packages/paimon-spark-3.4-0.8.2.jar,/tmp/gravitino/spark/packages/paimon-core-0.8.2.jar") \
    .config("spark.sql.catalog.paimon", "org.apache.paimon.spark.SparkCatalog") \
    .config("spark.sql.catalog.paimon.type", "paimon") \
    .config("spark.sql.catalog.paimon.metastore", "hive") \
    .config("spark.sql.catalog.paimon.uri", "thrift://hive:9083") \
    .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

print("SparkSession successfully created!")

SparkSession successfully created!


In [8]:
spark.sql("drop table paimon.mydatabase.employee;")
spark.sql("drop database paimon.mydatabase;")

DataFrame[]

In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS paimon.mydatabase")
print(spark.sql("SHOW DATABASES IN paimon").show())

+----------+
| namespace|
+----------+
|   default|
|      hudi|
|mydatabase|
|     sales|
+----------+

None


In [3]:
spark.sql("""
CREATE TABLE IF NOT EXISTS paimon.mydatabase.employee (
id BIGINT,
name STRING,
department STRING,
hire_date TIMESTAMP
)
USING paimon
PARTITIONED BY (name)
TBLPROPERTIES (
'metastore'='hive'
)
LOCATION 'hdfs://hive:9000/user/hive/warehouse/mydatabase/employee'
""")
spark.sql("SHOW TABLES IN paimon.mydatabase").show()

+----------+---------+-----------+
| namespace|tableName|isTemporary|
+----------+---------+-----------+
|mydatabase| employee|      false|
+----------+---------+-----------+



In [4]:
spark.sql("INSERT INTO paimon.mydatabase.employee VALUES (1, 'Alice', 'engineering', current_timestamp())")
spark.sql("INSERT INTO paimon.mydatabase.employee VALUES (2, 'Bob', 'hr', current_timestamp())")

DataFrame[]

In [5]:
spark.sql("SELECT * FROM paimon.mydatabase.employee;").show()

+---+-----+-----------+--------------------+
| id| name| department|           hire_date|
+---+-----+-----------+--------------------+
|  1|Alice|engineering|2025-09-12 03:40:...|
|  1|Alice|engineering|2025-09-12 04:10:...|
|  2|  Bob|         hr|2025-09-12 03:40:...|
|  2|  Bob|         hr|2025-09-12 04:10:...|
+---+-----+-----------+--------------------+



In [14]:
spark.sql("DROP TABLE paimon.mydatabase.employee;")

DataFrame[]

In [1]:
spark.top()

NameError: name 'spark' is not defined